# Init

In [0]:
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType
import pyspark.sql.functions as F 

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

# Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

# Data Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


In [0]:
df.display()

## Normalization

In [0]:
df = (
    df 
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
        .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
        .when(F.upper(F.col("cst_gndr")) == "M", "Male")
        .otherwise("n/a")
    )
)

# Renaming The Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


# Write Into Silver Table

In [0]:
(
    df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.cust_customers")
)